# Instagram 3차 정규화 CSV 컬럼명 변경

3차 정규화 CSV 원본을 `02_csv/02_instagram/temp`에 보관하고, 지정된 컬럼명을 변경한 CSV를 원래 경로에 같은 파일명으로 생성합니다.

In [ ]:
from pathlib import Path
import csv
import os
import shutil

COLUMN_RENAME = {
    "id": "post_id",
    "comments_count": "comment_count",
    "timestamp": "created_at",
    "caption": "description",
    "permalink": "post_url",
}


def find_project_root(start: Path) -> Path:
    """현재 작업 경로 또는 상위 경로에서 프로젝트 루트를 찾습니다."""
    for candidate in (start, *start.parents):
        if (candidate / "02_csv" / "02_instagram").is_dir():
            return candidate
    raise FileNotFoundError("02_csv/02_instagram 경로를 찾을 수 없습니다.")


def read_header(path: Path) -> list[str]:
    with path.open("r", encoding="utf-8-sig", newline="") as file:
        return next(csv.reader(file))


project_root = find_project_root(Path.cwd().resolve())
csv_dir = project_root / "02_csv" / "02_instagram"
temp_dir = csv_dir / "temp"
temp_dir.mkdir(exist_ok=True)

source_files = sorted(csv_dir.glob("instagram_normalized_3nf_*.csv"))
if not source_files:
    raise FileNotFoundError("3차 정규화 CSV 파일을 찾을 수 없습니다.")

backup_files = [temp_dir / path.name for path in source_files]
backup_exists = [path.exists() for path in backup_files]

if all(backup_exists):
    # 이미 처리된 상태인지 검증하고, 원본 백업을 덮어쓰지 않습니다.
    for output_path in source_files:
        actual_header = read_header(output_path)
        if any(old_name in actual_header for old_name in COLUMN_RENAME):
            raise RuntimeError(f"백업은 있지만 컬럼 변경이 완료되지 않았습니다: {output_path.name}")
    print("이미 컬럼명 변경이 완료되어 재처리를 건너뜁니다.")
elif any(backup_exists):
    raise RuntimeError("temp 폴더에 일부 원본만 존재합니다. 파일 상태를 확인해 주세요.")
else:
    # 원본 3NF CSV를 temp 폴더로 이동합니다.
    for source_path, backup_path in zip(source_files, backup_files):
        shutil.move(str(source_path), str(backup_path))

    results = []
    for backup_path in backup_files:
        output_path = csv_dir / backup_path.name
        temporary_output = csv_dir / f".{backup_path.name}.tmp"

        try:
            with (
                backup_path.open("r", encoding="utf-8-sig", newline="") as source_file,
                temporary_output.open("w", encoding="utf-8-sig", newline="") as output_file,
            ):
                reader = csv.reader(source_file)
                writer = csv.writer(output_file)

                original_header = next(reader)
                renamed_header = [COLUMN_RENAME.get(column, column) for column in original_header]
                if len(renamed_header) != len(set(renamed_header)):
                    raise ValueError(f"컬럼명 변경 후 중복 컬럼이 발생합니다: {backup_path.name}")

                writer.writerow(renamed_header)
                writer.writerows(reader)

            os.replace(temporary_output, output_path)
            changed_columns = {
                old: new for old, new in COLUMN_RENAME.items() if old in original_header
            }
            results.append((output_path.name, changed_columns))
        finally:
            if temporary_output.exists():
                temporary_output.unlink()

    for file_name, changed_columns in results:
        print(f"{file_name}: {changed_columns or '변경 대상 컬럼 없음'}")
